In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load cleaned dataset
df = pd.read_csv('../data/cleaned_news.csv')

# Remove empty rows
df = df.dropna(subset=['clean_title', 'clean_text'])

print("Data loaded!")
print(f"Total articles: {len(df)}")
print(f"Fake: {len(df[df['label'] == 'FAKE'])}")
print(f"Real: {len(df[df['label'] == 'REAL'])}")


Data loaded!
Total articles: 44266
Fake: 22850
Real: 21416


In [3]:
df['combined'] = df['clean_title'] + " " + df['clean_text']

X = df['combined']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Data prepared!")
print(f"Training articles: {len(X_train)}")
print(f"Testing articles: {len(X_test)}")

Data prepared!
Training articles: 35412
Testing articles: 8854


In [4]:
print("Converting text to numbers...")

vectorizer = TfidfVectorizer(max_features=10000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Text converted to numbers!")
print(f"Each article is now: {X_train_tfidf.shape[1]} numbers")

Converting text to numbers...
Text converted to numbers!
Each article is now: 10000 numbers


In [5]:
print("Training AI model...")
print("Please wait...")

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("AI model trained!")

Training AI model...
Please wait...
AI model trained!


In [6]:
predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print("AI MODEL RESULTS")
print("=" * 40)
print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, predictions))

AI MODEL RESULTS
Accuracy: 98.76%

Detailed Report:
              precision    recall  f1-score   support

        FAKE       0.99      0.99      0.99      4567
        REAL       0.99      0.99      0.99      4287

    accuracy                           0.99      8854
   macro avg       0.99      0.99      0.99      8854
weighted avg       0.99      0.99      0.99      8854



In [7]:
def predict_news(headline):
    import re
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    from nltk.stem import WordNetLemmatizer
    
    text = headline.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(w) for w in words]
    clean = " ".join(words)
    
    text_tfidf = vectorizer.transform([clean])
    prediction = model.predict(text_tfidf)[0]
    probability = model.predict_proba(text_tfidf)[0]
    confidence = max(probability) * 100
    
    print("=" * 50)
    print(f"Headline: {headline}")
    print(f"Prediction: {prediction}")
    print(f"Confidence: {confidence:.2f}%")
    print("=" * 50)

predict_news("SHOCKING: Scientists PROVE earth is flat!!!")
predict_news("Federal Reserve raises interest rates by 0.25%")
predict_news("MIRACLE cure for cancer BANNED by doctors!!!")
predict_news("United Nations releases new climate change report")

Headline: SHOCKING: Scientists PROVE earth is flat!!!
Prediction: FAKE
Confidence: 89.01%
Headline: Federal Reserve raises interest rates by 0.25%
Prediction: FAKE
Confidence: 55.34%
Headline: MIRACLE cure for cancer BANNED by doctors!!!
Prediction: FAKE
Confidence: 71.97%
Headline: United Nations releases new climate change report
Prediction: FAKE
Confidence: 86.83%


In [8]:
def predict_news(text):
    import re
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    from nltk.stem import WordNetLemmatizer
    
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]
    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(w) for w in words]
    clean = " ".join(words)
    
    text_tfidf = vectorizer.transform([clean])
    prediction = model.predict(text_tfidf)[0]
    probability = model.predict_proba(text_tfidf)[0]
    confidence = max(probability) * 100
    
    print("=" * 50)
    print(f"Prediction: {prediction}")
    print(f"Confidence: {confidence:.2f}%")
    print("=" * 50)

# Test with longer real news text
predict_news("""
The Federal Reserve raised its benchmark interest rate 
by a quarter percentage point on Wednesday, the central 
bank said in a statement. The decision was unanimous among 
voting members of the Federal Open Market Committee. 
The rate increase brings the federal funds rate to a 
target range of 5.25 to 5.5 percent.
""")

# Test with longer fake news text
predict_news("""
SHOCKING revelation as scientists finally admit the earth 
is completely flat and NASA has been hiding this secret 
for decades. Multiple whistleblowers have come forward 
with proof that the moon landing was also completely fake. 
The government does not want you to know this truth.
""")

Prediction: REAL
Confidence: 93.75%
Prediction: FAKE
Confidence: 97.32%


In [10]:
import pickle
import os

# Save the model
with open('../models/fake_news_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the vectorizer
with open('../models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model saved successfully!")
print("Saved to: models/fake_news_model.pkl")
print("Saved to: models/vectorizer.pkl")
print("")
print("PHASE 4 COMPLETE!")
print("You just built a real AI that is 98.76% accurate!")

Model saved successfully!
Saved to: models/fake_news_model.pkl
Saved to: models/vectorizer.pkl

PHASE 4 COMPLETE!
You just built a real AI that is 98.76% accurate!
